In [2]:
import os 
import json 
import pandas as pd 

from dotenv import load_dotenv 
import google.generativeai as genai 
import streamlit as st 

c:\Users\Quang Minh\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('./data/horoscope_saved.csv')
df.head()

,sign,category,date,horoscope
0,aries,general,20200617,"There's a great day ahead of you, Aries. You'l..."
1,aries,general,20200618,People will understand and appreciate your des...
2,aries,general,20200619,You are very interested in technological break...
3,aries,general,20200620,Stress from overwork could have you feeling we...
4,aries,general,20200621,This is a good day to stand up for yourself an...


In [3]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df

,sign,category,date,horoscope
0,aries,general,2020-06-17,"There's a great day ahead of you, Aries. You'l..."
1,aries,general,2020-06-18,People will understand and appreciate your des...
2,aries,general,2020-06-19,You are very interested in technological break...
3,aries,general,2020-06-20,Stress from overwork could have you feeling we...
4,aries,general,2020-06-21,This is a good day to stand up for yourself an...
...,...,...,...,...
21954,pisces,birthday,2021-06-12,Celebrate in style on your birthday in prepara...
21955,pisces,birthday,2021-06-13,Imagine your life as if it was exactly the way...
21956,pisces,birthday,2021-06-14,"Fun, playfulness, and humor are easy to manife..."
21957,pisces,birthday,2021-06-15,"Your birthday brings you a fresh start, as you..."


In [4]:
df.isna().sum()

sign         0
category     0
date         0
horoscope    0
dtype: int64

In [5]:
df.nunique()

sign            12
category         5
date           366
horoscope    12050
dtype: int64

In [6]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df.to_json(orient='records')
df

,sign,category,date,horoscope,day_of_week,month,year
0,aries,general,2020-06-17,"There's a great day ahead of you, Aries. You'l...",2,6,2020
1,aries,general,2020-06-18,People will understand and appreciate your des...,3,6,2020
2,aries,general,2020-06-19,You are very interested in technological break...,4,6,2020
3,aries,general,2020-06-20,Stress from overwork could have you feeling we...,5,6,2020
4,aries,general,2020-06-21,This is a good day to stand up for yourself an...,6,6,2020
...,...,...,...,...,...,...,...
21954,pisces,birthday,2021-06-12,Celebrate in style on your birthday in prepara...,5,6,2021
21955,pisces,birthday,2021-06-13,Imagine your life as if it was exactly the way...,6,6,2021
21956,pisces,birthday,2021-06-14,"Fun, playfulness, and humor are easy to manife...",0,6,2021
21957,pisces,birthday,2021-06-15,"Your birthday brings you a fresh start, as you...",1,6,2021


In [3]:
import requests
def get_zodiac_daily(sign):
    api_url = f'https://api.api-ninjas.com/v1/horoscope?zodiac={sign}'
    response = requests.get(api_url, headers={'X-Api-Key': st.secrets['api']['horoscope-api-key']})
    if response.status_code == requests.codes.ok:
        return response.json()
    else:
        return {"error": response.status_code, "message": response.text}
get_zodiac_daily('ARIES')

{'date': '2025-07-05',
 'sign': 'Aries',
 'horoscope': "A cherished person in your life might voice their dissatisfaction with the current situation, Aries. You may sense some underlying tension brewing. Be cautious, as emotions could flare up. There's a strong urge for movement and change, a restlessness that calls for you to take initiative. However, proceed with caution, as your efforts may encounter resistance if you're not mindful."}

In [5]:
zodiac_signs = [
    "aries", "taurus", "gemini", "cancer", "leo", "virgo",
    "libra", "scorpio", "sagittarius", "capricorn", "aquarius", "pisces"
]

def extract_zodiac_keywords(text):
    words = text.lower().split()
    return [sign for sign in zodiac_signs if sign in words]

# Example
user_input = "I'm a Leo sun, Aries moon, and Gemini rising"
print(extract_zodiac_keywords(user_input))
# Output: ['leo', 'pisces', 'gemini']


['aries', 'gemini', 'leo']


In [13]:
load_dotenv()
google_api_key = st.secrets['api']['api-key']
genai.configure(api_key=google_api_key)

In [14]:
model = genai.GenerativeModel("gemini-1.5-flash") #Gemini 1.5 Flash

In [15]:
prompt = "Bạn là ai?"
res = model.generate_content(prompt)
res

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "T\u00f4i l\u00e0 m\u1ed9t m\u00f4 h\u00ecnh ng\u00f4n ng\u1eef l\u1edbn, \u0111\u01b0\u1ee3c hu\u1ea5n luy\u1ec7n b\u1edfi Google."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "avg_logprobs": -0.1621614545583725
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 4,
        "candidates_token_count": 16,
        "total_token_count": 20
      },
      "model_version": "gemini-1.5-flash"
    }),
)

In [16]:
res.text

'Tôi là một mô hình ngôn ngữ lớn, được huấn luyện bởi Google.'

In [25]:
zodiac_df = df

with open('config.json','r') as f :
    config = json.load(f)
    functions = config.get('functions')
    
genai.GenerativeModel("gemini-1.5-flash",
                            system_instruction=f"""
                                You are an expert astrologer named AstroBot Your role is to provide friendly, insightful, and personalized astrological guidance to users.

                                You can:
                                - Interpret zodiac signs (sun, moon, rising)
                                - Explain astrological placements (e.g., Mercury in Gemini)
                                - Offer daily, weekly, or monthly horoscopes
                                - Provide compatibility insights between signs
                                - Suggest affirmations, journaling prompts, and self-care tips based on astrology
                                - Answer spiritual or emotional questions through a cosmic lens
                                - Give today's horoscope for a specific zodiac sign through function get_zodiac_daily(sign), return the response in JSON format
                                    method get_zodiac_daily(sign)
                                    1. get today's date
                                    2. get zodiac sign from user input
                                    3. call API to get horoscope data
                                    4. return the response in JSON format

                                Always personalize your response when the user gives:
                                - Birthday, birth time, and birth location (use it to determine sign traits)
                                - Sun, moon, or rising sign
                                - The other person's sign (for compatibility)

                                Avoid long lectures. Break down complex terms in simple language. Use metaphors, emojis, or poetic phrases when appropriate. For example:

                                - “You're a Cancer Rising — people feel your energy before you speak. It's soft, intuitive, and quietly powerful 🌊.”
                                - “With Venus in Scorpio, your love runs deep. You don't do casual — you do cosmic connections.”
                                - “A Gemini Moon? Your emotions ride on breezes — curious, quick, and constantly shifting.”

                                If you're unsure about birth chart data (e.g., if the user doesn't know birth time), make a helpful guess and explain why time matters.

                                NEVER claim to predict the future. Instead, guide users with insight, encouragement, and cosmic curiosity.

                                Gently redirect unrelated questions (e.g., coding, math) with something like:
                                > "My stars are better aligned for love, signs, and soul-searching than for math problems — shall we explore your chart instead?"

                                End your answers with warmth or a gentle affirmation like:
                                ✨ "May the stars light your path." ✨
                                    """)

genai.GenerativeModel(
    model_name='models/gemini-1.5-flash',
    generation_config={},
    safety_settings={},
    tools=None,
    system_instruction='\n                                You are an expert astrologer named AstroBot Your role is to provide friendly, insightful, and personalized astrological guidance to users.\n\n                                You can:\n                                - Interpret zodiac signs (sun, moon, rising)\n                                - Explain astrological placements (e.g., Mercury in Gemini)\n                                - Offer daily, weekly, or monthly horoscopes\n                                - Provide compatibility insights between signs\n                                - Suggest affirmations, journaling prompts, and self-care tips based on astrology\n                                - Answer spiritual or emotional questions through a cosmic lens\n                                - Give today\'s horoscope for a specific zodiac sign t

In [26]:
prompt = "cung hoàng đạo của tôi là Bạch Dương. Hãy cho tôi biết về vận mệnh của tôi hôm nay."
response = model.generate_content(prompt)
response.text

'Hôm nay, vận mệnh của bạn, Bạch Dương, tràn đầy năng lượng và sự nhiệt tình. Bạn sẽ có một ngày tràn đầy hoạt động và cơ hội để thể hiện sự dũng cảm và chủ động của mình.  Tuy nhiên, hãy cẩn thận với xu hướng nóng nảy và thiếu kiên nhẫn vốn có. Hãy cố gắng kiềm chế cảm xúc để tránh những xung đột không đáng có.\n\n**Về công việc:**  Bạn sẽ tràn đầy sáng tạo và năng lượng để giải quyết những thách thức.  Đây là thời điểm lý tưởng để bắt đầu một dự án mới hoặc thúc đẩy những kế hoạch đã ấp ủ.  Nhưng hãy nhớ lên kế hoạch chi tiết và tránh hành động hấp tấp. Sự chuẩn bị kỹ càng sẽ giúp bạn đạt được hiệu quả cao hơn.\n\n**Về tình cảm:**  Nếu bạn đang độc thân, hãy mở lòng mình và đón nhận những cơ hội mới. Sự tự tin và năng lượng tích cực của bạn sẽ thu hút sự chú ý của người khác.  Nếu bạn đang trong mối quan hệ, hãy dành thời gian chất lượng cho người yêu và thể hiện tình cảm chân thành của mình.  Tuy nhiên, hãy tránh những cuộc tranh cãi không cần thiết.\n\n**Về sức khỏe:**  Hãy giữ cho